# NVIDIA NIM Speech & Audio Models — Direct Test Notebook

Calls all 3 NIM models directly — **no project code**. All use gRPC.

| Model | Protocol | NVCF Function ID |
|-------|----------|-----------------|
| **BNR (Denoise)** | gRPC | `66518fde-1164-479b-a21f-f8240104505a` |
| **Canary-1B ASR** | gRPC | `b0e8b4a5-217c-40b7-9b96-17d84e666317` |
| **Magpie TTS Zero-shot** | gRPC | `4e813649-d5e4-4020-b2be-2b918396d19d` |

All use `grpc.nvcf.nvidia.com:443` with TLS + metadata auth.

**Dependencies:**
```
pip install nvidia-riva-client grpcio protobuf ipywidgets sounddevice soundfile
```

> Magpie TTS Zero-shot is **gated** — you need approved access on build.nvidia.com.

## 0. Setup — API Key & Auth Helpers

In [ ]:
import os, sys, io, tempfile
from pathlib import Path
from IPython.display import Audio, display

# ── Function IDs ──
FID_BNR = "66518fde-1164-479b-a21f-f8240104505a"
FID_ASR = "b0e8b4a5-217c-40b7-9b96-17d84e666317"
FID_TTS = "4e813649-d5e4-4020-b2be-2b918396d19d"

# ── Read API key ──
API_KEY = os.getenv("NVIDIA_API_KEY", "")
if not API_KEY:
    env_path = Path("backend/.env")
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            if line.startswith("NVIDIA_API_KEY="):
                API_KEY = line.split("=", 1)[1].strip().strip('"').strip("'")
                break

print(f"API Key: {'nvapi-...' + API_KEY[-4:] if API_KEY else 'NOT SET'}")
print(f"ASR FID : {FID_ASR}")
print(f"TTS FID : {FID_TTS}")
print(f"BNR FID : {FID_BNR}")

In [ ]:
import riva.client

# ── Common metadata for all NVCF gRPC calls ──
def gRPC_metadata(function_id: str) -> list:
    return [
        ("function-id", function_id),
        ("authorization", f"Bearer {API_KEY}"),
    ]


def asr_auth():
    return riva.client.Auth(
        use_ssl=True, uri="grpc.nvcf.nvidia.com:443",
        metadata_args=gRPC_metadata(FID_ASR),
    )


def tts_auth():
    return riva.client.Auth(
        use_ssl=True, uri="grpc.nvcf.nvidia.com:443",
        metadata_args=gRPC_metadata(FID_TTS),
        options=[("grpc.max_receive_message_length", 20 * 1024 * 1024)],
    )


# ── File I/O helpers ──
def load_audio(path: str) -> bytes:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Not found: {p.resolve()}")
    data = p.read_bytes()
    print(f"Loaded {len(data):,} bytes ← {p.name}")
    return data


def save_audio(data: bytes, path: str) -> str:
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_bytes(data)
    print(f"Saved {len(data):,} bytes → {out.resolve()}")
    return str(out.resolve())


print("Ready. nvidia-riva-client imported.")

---
## 1. BNR — Background Noise Removal (gRPC)

Uses `bnr_grpc.py` (embedded Maxine proto). Transactional mode — sends whole WAV.
Sample rate must be 16000 or 48000 Hz.

In [ ]:
from bnr_grpc import bnr_denoise

noisy = load_audio("backend/uploads/recordings/sample.wav")  # ← change

display(Audio(noisy, autoplay=False))
print("^ Original (noisy)")

cleaned = bnr_denoise(
    api_key=API_KEY,
    function_id=FID_BNR,
    wav_bytes=noisy,
    sample_rate=48000,
    intensity_ratio=1.0,
)

save_audio(cleaned, "/tmp/bnr_cleaned.wav")
display(Audio(cleaned, autoplay=False))
print("^ Cleaned")

---
## 2. ASR Transcription (gRPC — Canary-1B)

**Service:** `RivaSpeechRecognition/Recognize` (offline)  
**Input:** WAV bytes + language code  
**Output:** `RecognizeResponse` with transcript

In [ ]:
audio = load_audio("backend/uploads/recordings/sample.wav")  # ← change

asr = riva.client.ASRService(asr_auth())
config = riva.client.RecognitionConfig(
    language_code="en-US",
    enable_automatic_punctuation=True,
)

print("→ ASR gRPC: grpc.nvcf.nvidia.com:443")
resp = asr.offline_recognize(audio, config)

for result in resp.results:
    for alt in result.alternatives:
        print(f"Transcript  : {alt.transcript}")
        print(f"Confidence  : {alt.confidence:.3f}")
        if alt.words:
            for w in alt.words[:10]:
                print(f"  [{w.start_time}–{w.end_time}ms] {w.word}")
        print()

---
## 3. ASR Translation (gRPC — Canary-1B)

Same service, add `custom_configuration` for translation.
`--custom-configuration "target_language:fr,task:translate"`

In [ ]:
audio = load_audio("backend/uploads/recordings/sample.wav")  # ← change

TARGET = "fr"  # try: es, de, zh, ja, ko, hi

asr = riva.client.ASRService(asr_auth())
config = riva.client.RecognitionConfig(
    language_code="en-US",
    enable_automatic_punctuation=True,
)
riva.client.add_custom_configuration_to_config(
    config, f"target_language:{TARGET},task:translate"
)

print(f"→ ASR Translate gRPC (→ {TARGET})")
resp = asr.offline_recognize(audio, config)

for result in resp.results:
    for alt in result.alternatives:
        print(f"Translated ({TARGET}): {alt.transcript}")
        if alt.language_code:
            print(f"Detected languages: {alt.language_code}")

---
## 4. Voice Cloning + TTS (gRPC — Magpie Zero-shot)

**Service:** `RivaSpeechSynthesis/Synthesize` (offline)  
**Input:** Reference voice WAV (3–10s) + text to speak  
**Output:** Synthesized speech WAV bytes  

> ⚠ Requires approved access to Magpie TTS on build.nvidia.com

In [ ]:
voice_bytes = load_audio("backend/uploads/voices/my_voice.wav")  # ← change

# Save to temp file — nvidia-riva-client needs a file path for the audio prompt
tmp_ref = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
tmp_ref.write(voice_bytes)
tmp_ref.close()

text = "Hello! This is my cloned voice. It sounds remarkably natural."

tts = riva.client.SpeechSynthesisService(tts_auth())

print(f"→ TTS gRPC: grpc.nvcf.nvidia.com:443")
print(f"  text: \"{text}\"")
print(f"  voice: {len(voice_bytes):,} bytes reference")

resp = tts.synthesize(
    text=text,
    voice="Magpie-ZeroShot.Speaker",
    language_code="en-US",
    sample_rate_hz=48000,
    zero_shot_audio_prompt_file=tmp_ref.name,
    zero_shot_quality=20,
)

# Clean up temp file
os.unlink(tmp_ref.name)

if resp.audio:
    save_audio(resp.audio, "/tmp/tts_cloned.wav")
    display(Audio(resp.audio, rate=48000, autoplay=False))
    print(f"^ Cloned voice: \"{text}\"")
else:
    print("TTS returned no audio — check access.")

---
## 5. Full Pipeline (all gRPC)

Chain: Denoise → Transcribe → Translate → Re-voice

In [ ]:
from bnr_grpc import bnr_denoise
import tempfile

INPUT = "backend/uploads/recordings/sample.wav"   # ← change
VOICE = "backend/uploads/voices/my_voice.wav"     # ← change

raw = load_audio(INPUT)

# ── 1. Denoise ──
print("\n── 1. BNR Denoise (gRPC) ──")
try:
    clean = bnr_denoise(API_KEY, FID_BNR, raw, sample_rate=48000)
except Exception as e:
    print(f"BNR failed: {e}, using raw")
    clean = raw

# ── 2. Transcribe ──
print("\n── 2. ASR Transcribe (gRPC) ──")
asr = riva.client.ASRService(asr_auth())
cfg = riva.client.RecognitionConfig(language_code="en-US", enable_automatic_punctuation=True)
resp = asr.offline_recognize(clean, cfg)
transcript = resp.results[0].alternatives[0].transcript if resp.results else ""
print(f"   → {transcript}")

# ── 3. Translate ──
print("\n── 3. ASR Translate (gRPC) ──")
cfg2 = riva.client.RecognitionConfig(language_code="en-US", enable_automatic_punctuation=True)
riva.client.add_custom_configuration_to_config(cfg2, "target_language:fr,task:translate")
xl = asr.offline_recognize(clean, cfg2)
print(f"   FR: {xl.results[0].alternatives[0].transcript if xl.results else 'N/A'}")

# ── 4. Re-voice ──
print("\n── 4. TTS Re-voice (gRPC) ──")
if Path(VOICE).exists() and transcript:
    v = load_audio(VOICE)
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    tmp.write(v); tmp.close()
    tts = riva.client.SpeechSynthesisService(tts_auth())
    tts_resp = tts.synthesize(
        text=transcript,
        voice="Magpie-ZeroShot.Speaker",
        language_code="en-US",
        sample_rate_hz=48000,
        zero_shot_audio_prompt_file=tmp.name,
        zero_shot_quality=20,
    )
    os.unlink(tmp.name)
    if tts_resp.audio:
        save_audio(tts_resp.audio, "/tmp/pipeline_output.wav")
        display(Audio(tts_resp.audio, rate=48000, autoplay=False))
        print("^ Re-voiced output")
else:
    print("   Skipped — no voice or transcript")

---
## 6. Interactive Recorder

Buttons to record, play back, and save to project dirs.
Requires: `pip install sounddevice soundfile ipywidgets`

In [ ]:
import sounddevice as sd
import soundfile as sf
import ipywidgets as widgets
from IPython.display import Audio, display, clear_output
from datetime import datetime
import threading

VOICES_DIR     = Path("backend/uploads/voices");     VOICES_DIR.mkdir(parents=True, exist_ok=True)
RECORDINGS_DIR = Path("backend/uploads/recordings"); RECORDINGS_DIR.mkdir(parents=True, exist_ok=True)
CLIPS_DIR      = Path("backend/uploads/clips");      CLIPS_DIR.mkdir(parents=True, exist_ok=True)

recorded_audio: bytes | None = None
last_sr: int = 48000

In [ ]:
status   = widgets.Output()
player   = widgets.Output()
filelist = widgets.Output()

dur = widgets.IntSlider(value=5, min=1, max=30, step=1, description="Duration (s):",
                        style={"description_width": "initial"})
sr  = widgets.Dropdown(options=[("48 kHz", 48000), ("16 kHz", 16000)], value=48000,
                       description="Sample rate:")
rec_btn = widgets.Button(description=" Record", icon="microphone",
                         button_style="danger", layout=widgets.Layout(width="120px"))
play_btn = widgets.Button(description=" Play", icon="play",
                          button_style="success", layout=widgets.Layout(width="120px"))

fname = widgets.Text(value="", placeholder="filename (no extension)",
                     description="Name:", layout=widgets.Layout(width="340px"))
target_dir = widgets.Dropdown(options=[("voices/", "voices"), ("recordings/", "recordings"),
                                      ("clips/", "clips")], value="recordings",
                             description="Save to:")
save_btn = widgets.Button(description="Save", icon="floppy-disk",
                          button_style="primary", layout=widgets.Layout(width="100px"))
refresh_btn = widgets.Button(description="Refresh file list", icon="refresh",
                             layout=widgets.Layout(width="160px"))

In [ ]:
def do_record(_):
    global recorded_audio, last_sr
    d, r = dur.value, sr.value
    last_sr = r
    rec_btn.disabled = play_btn.disabled = save_btn.disabled = True
    with status:
        clear_output()
        print(f"Recording {d}s at {r}Hz...")

    def _record():
        global recorded_audio
        try:
            audio = sd.rec(int(d * r), samplerate=r, channels=1, dtype="int16")
            sd.wait()
            buf = io.BytesIO()
            sf.write(buf, audio, r, format="WAV")
            recorded_audio = buf.getvalue()
            with status:
                clear_output()
                print(f"Done — {len(recorded_audio):,} bytes")
            with player:
                clear_output()
                display(Audio(recorded_audio, rate=r, autoplay=False))
        except Exception as e:
            with status:
                clear_output()
                print(f"Error: {e}")
        finally:
            rec_btn.disabled = play_btn.disabled = save_btn.disabled = False

    threading.Thread(target=_record, daemon=True).start()


def do_play(_):
    if recorded_audio is None:
        with status: clear_output(); print("Nothing recorded.")
        return
    with player:
        clear_output()
        display(Audio(recorded_audio, rate=last_sr, autoplay=False))


def do_save(_):
    if recorded_audio is None:
        with status: clear_output(); print("Record first.")
        return
    name = fname.value.strip() or datetime.now().strftime("rec_%Y%m%d_%H%M%S")
    dirs = {"voices": VOICES_DIR, "recordings": RECORDINGS_DIR, "clips": CLIPS_DIR}
    path = dirs[target_dir.value] / f"{name}.wav"
    path.write_bytes(recorded_audio)
    with status: clear_output(); print(f"Saved → {path}")
    refresh_files(None)


def refresh_files(_):
    with filelist:
        clear_output()
        for label, d in [("Voices", VOICES_DIR), ("Recordings", RECORDINGS_DIR), ("Clips", CLIPS_DIR)]:
            files = sorted(d.glob("*.wav")) + sorted(d.glob("*.mp3"))
            print(f"── {label} ──")
            for f in files:
                print(f"  {f.name}  ({f.stat().st_size:,} bytes)")
            if not files:
                print("  (empty)")
            print()


rec_btn.on_click(do_record)
play_btn.on_click(do_play)
save_btn.on_click(do_save)
refresh_btn.on_click(refresh_files)
refresh_files(None)

In [ ]:
# ── Render the recorder UI ──
display(widgets.HBox([dur, sr]))
display(widgets.HBox([rec_btn, play_btn]))
display(status)
display(player)
display(widgets.HBox([fname, target_dir, save_btn]))
display(refresh_btn)
display(filelist)